In [ ]:
# ============================================================
# LIONS CAMP SCRAPER — Google Colab
# Scrape → filtre 2026 → extrait PDF à la volée → JSON
# Site italien → pays extraits de l'URL, traduits EN
# Régions détaillées : North / Middle / South America
# USA : country + state depuis l'URL
# ============================================================

# ── 1. Installation ──────────────────────────────────────────
!pip install requests beautifulsoup4 pdfplumber -q

# ── 2. Imports ───────────────────────────────────────────────
import requests, json, re, os, io
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from IPython.display import display
import pdfplumber
import pandas as pd

# ── 3. Config ────────────────────────────────────────────────
BASE_URL    = "https://www.scambigiovanili-lions.org/campi-lions-esteri-nel-mondo"
BASE_DOMAIN = "https://www.scambigiovanili-lions.org"
URL_PREFIX_SEGMENTS = {"images", "doc-campi-esteri", "doc-campi-italia"}

OUTPUT_JSON = "lions_camps_2026.json"
PDF_FOLDER  = "/content/pdf_camps"
TARGET_YEAR = "2026"

os.makedirs(PDF_FOLDER, exist_ok=True)

# ── 4. Dictionnaire Italien → Anglais ────────────────────────
ITALIAN_TO_ENGLISH = {
    # Europa
    "austria"               : "Austria",
    "belgio"                : "Belgium",
    "danimarca"             : "Denmark",
    "finlandia"             : "Finland",
    "francia"               : "France",
    "germania"              : "Germany",
    "gran bretagna"         : "United Kingdom",
    "regno unito"           : "United Kingdom",
    "uk"                    : "United Kingdom",
    "grecia"                : "Greece",
    "irlanda"               : "Ireland",
    "islanda"               : "Iceland",
    "italia"                : "Italy",
    "campi italia"          : "Italy",
    "lussemburgo"           : "Luxembourg",
    "malta"                 : "Malta",
    "norvegia"              : "Norway",
    "olanda"                : "Netherlands",
    "paesi bassi"           : "Netherlands",
    "polonia"               : "Poland",
    "portogallo"            : "Portugal",
    "repubblica ceca"       : "Czech Republic",
    "romania"               : "Romania",
    "slovacchia"            : "Slovakia",
    "slovenia"              : "Slovenia",
    "spagna"                : "Spain",
    "svezia"                : "Sweden",
    "svizzera"              : "Switzerland",
    "turchia"               : "Turkey",
    "ucraina"               : "Ukraine",
    "ungheria"              : "Hungary",
    "croazia"               : "Croatia",
    "serbia"                : "Serbia",
    "bulgaria"              : "Bulgaria",
    "estonia"               : "Estonia",
    "lettonia"              : "Latvia",
    "lituania"              : "Lithuania",
    # Nord America
    "canada"                : "Canada",
    "stati-uniti"           : "United States",
    "stati uniti"           : "United States",
    "usa"                   : "United States",
    # America Centrale
    "costa-rica"            : "Costa Rica",
    "costa rica"            : "Costa Rica",
    "cuba"                  : "Cuba",
    "guatemala"             : "Guatemala",
    "honduras"              : "Honduras",
    "messico"               : "Mexico",
    "panama"                : "Panama",
    "repubblica-dominicana" : "Dominican Republic",
    "repubblica dominicana" : "Dominican Republic",
    # Sud America
    "argentina"             : "Argentina",
    "bolivia"               : "Bolivia",
    "brasile"               : "Brazil",
    "cile"                  : "Chile",
    "colombia"              : "Colombia",
    "ecuador"               : "Ecuador",
    "paraguay"              : "Paraguay",
    "perù"                  : "Peru",
    "peru"                  : "Peru",
    "uruguay"               : "Uruguay",
    "venezuela"             : "Venezuela",
    # Asia-Pacifico
    "australia"             : "Australia",
    "cina"                  : "China",
    "mongolia"              : "Mongolia",
    "giappone"              : "Japan",
    "india"                 : "India",
    "indonesia"             : "Indonesia",
    "nuova-zelanda"         : "New Zealand",
    "nuova zelanda"         : "New Zealand",
    "corea-del-sud"         : "South Korea",
    "corea del sud"         : "South Korea",
    "taiwan"                : "Taiwan",
    "thailandia"            : "Thailand",
    "filippine"             : "Philippines",
    "vietnam"               : "Vietnam",
    # Africa
    "egitto"                : "Egypt",
    "etiopia"               : "Ethiopia",
    "ghana"                 : "Ghana",
    "kenya"                 : "Kenya",
    "marocco"               : "Morocco",
    "nigeria"               : "Nigeria",
    "sudafrica"             : "South Africa",
    "sud-africa"            : "South Africa",
    "tanzania"              : "Tanzania",
    "tunisia"               : "Tunisia",
    # Medio Oriente
    "emirati-arabi"         : "UAE",
    "emirati arabi"         : "UAE",
    "giordania"             : "Jordan",
    "israele"               : "Israel",
    "libano"                : "Lebanon",
}

# ── 5. Pays → Région ─────────────────────────────────────────
COUNTRY_REGION = {
    # Europe
    "Austria":"Europe","Belgium":"Europe","Denmark":"Europe",
    "Finland":"Europe","France":"Europe","Germany":"Europe",
    "United Kingdom":"Europe","Greece":"Europe","Ireland":"Europe",
    "Iceland":"Europe","Italy":"Europe","Luxembourg":"Europe","Malta":"Europe",
    "Norway":"Europe","Netherlands":"Europe","Poland":"Europe",
    "Portugal":"Europe","Czech Republic":"Europe","Romania":"Europe",
    "Slovakia":"Europe","Slovenia":"Europe","Spain":"Europe",
    "Sweden":"Europe","Switzerland":"Europe","Turkey":"Europe",
    "Ukraine":"Europe","Hungary":"Europe","Croatia":"Europe",
    "Serbia":"Europe","Bulgaria":"Europe","Estonia":"Europe",
    "Latvia":"Europe","Lithuania":"Europe",
    # North America
    "Canada":"North America",
    "United States":"North America",
    # Middle America
    "Costa Rica":"Middle America","Cuba":"Middle America",
    "Guatemala":"Middle America","Honduras":"Middle America",
    "Mexico":"Middle America","Panama":"Middle America",
    "Dominican Republic":"Middle America",
    # South America
    "Argentina":"South America","Bolivia":"South America",
    "Brazil":"South America","Chile":"South America",
    "Colombia":"South America","Ecuador":"South America",
    "Paraguay":"South America","Peru":"South America",
    "Uruguay":"South America","Venezuela":"South America",
    # Asia-Pacific
    "Australia":"Asia-Pacific","China":"Asia-Pacific","Nepal":"Asia-Pacific",
    "Japan":"Asia-Pacific","India":"Asia-Pacific","Sri Lanka":"Asia-Pacific",
    "Indonesia":"Asia-Pacific","New Zealand":"Asia-Pacific","Mongolia":"Asia-Pacific",
    "South Korea":"Asia-Pacific","Taiwan":"Asia-Pacific",
    "Thailand":"Asia-Pacific","Philippines":"Asia-Pacific",
    "Vietnam":"Asia-Pacific",
    # Africa
    "Egypt":"Africa","Ethiopia":"Africa","Ghana":"Africa",
    "Kenya":"Africa","Morocco":"Africa","Namibia":"Africa", "Nigeria":"Africa",
    "South Africa":"Africa","Tanzania":"Africa","Tunisia":"Africa",
    # Middle East
    "UAE":"Middle East","Jordan":"Middle East",
    "Israel":"Middle East","Lebanon":"Middle East",
}

# ── 6. Traduction nom de pays ─────────────────────────────────
def translate_country(raw_slug):
    """
    raw_slug : segment d'URL brut, ex. 'stati-uniti', 'gran-bretagna'
    Essaie d'abord le slug exact, puis le slug avec espaces,
    puis matching partiel, sinon retourne le slug capitalisé.
    """
    slug_dash  = raw_slug.lower().strip()          # 'stati-uniti'
    slug_space = slug_dash.replace("-", " ")       # 'stati uniti'

    for key in (slug_dash, slug_space):
        if key in ITALIAN_TO_ENGLISH:
            return ITALIAN_TO_ENGLISH[key]

    # Matching partiel
    for ita, eng in ITALIAN_TO_ENGLISH.items():
        if ita in slug_space:
            return eng

    return slug_dash.replace("-", " ").title()     # fallback

# ── 7. Extraction pays + région + state + année depuis l'URL ──
#
#  Structure réelle :
#    /images/doc-campi-esteri/country/year/camp.pdf
#    /images/doc-campi-esteri/stati-uniti/state/year/camp.pdf
#
URL_PREFIX_SEGMENTS = {"images", "doc-campi-esteri", "doc-campi-italia"}

def fix_year(raw_year):
    """Corrige les typos d'année fréquentes ex: 2926 → 2026."""
    if not raw_year or not raw_year.isdigit():
        return None
    YEAR_FIXES = {
        "2926": "2026",
        "2016": "2026",  # typo possible
        "2062": "2026",  # inversion chiffres
        "2206": "2026",  # inversion chiffres
    }
    if raw_year in YEAR_FIXES:
        print(f"    ⚠️  Typo année corrigée : {raw_year} → {YEAR_FIXES[raw_year]}")
        return YEAR_FIXES[raw_year]
    return raw_year

def parse_link_text(link_text, pdf_name):
    """Extrait année et camp_name depuis le texte du lien HTML.

    Formats attendus :
      "2026 - Lionscamp Sound of Music"
      "2026 Lionscamp Sound of Music"
      "Lionscamp Sound of Music"   ← fallback sans année
    """
    text = link_text.strip()

    # Cherche une année 202x en début de chaîne
    match = re.match(r"^(202\d)\s*[-–:]?\s*(.+)$", text)
    if match:
        year_from_link = fix_year(match.group(1))
        camp_name      = match.group(2).strip()
    else:
        year_from_link = None
        camp_name      = text if text else pdf_name.replace(".pdf", "")

    return camp_name, year_from_link

def guess_country_region(pdf_url):
    parsed = urlparse(pdf_url)

    # Segments non vides, on retire le préfixe connu
    parts = [
        p for p in parsed.path.split("/")
        if p.strip() and p.lower() not in URL_PREFIX_SEGMENTS
    ]
    # Standard : parts = [country, year, camp.pdf]
    # USA      : parts = [country, state, year, camp.pdf]

    raw_country = parts[0] if len(parts) > 0 else ""
    country     = translate_country(raw_country)
    region      = COUNTRY_REGION.get(country, "Other")
    state       = None
    year        = None

    if country == "United States" or country == "Canada" and len(parts) >= 4:
        # USA, Canada : country / state / year / camp.pdf
        raw_state = parts[1].replace("-", " ").replace("_", " ")
        state     = raw_state.strip().title()
        year      = parts[2] if parts[2].isdigit() else None
        year      = fix_year(year)
    elif country == "Italy":
        year      = none
    elif len(parts) >= 2:
        # Standard : country / year / camp.pdf
        year      = parts[1] if parts[1].isdigit() else None
        year      = fix_year(year)

    return country, region, state, year

# ── 8. Extraction texte PDF depuis bytes ─────────────────────
def extract_text_from_bytes(pdf_bytes):
    raw_text = ""
    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                raw_text += t + "\n"
    return raw_text

# ── 9. Détection année dans le texte ─────────────────────────
def extract_year_from_pdf(raw_text):
    years = re.findall(r"\b(202[0-9])\b", raw_text)
    return years[0] if years else None

# ── 10. Parsing champs Lions Camp ─────────────────────────────
FIELD_MAP = {
    "Camp Location"        : "camp_location",
    "From/to"              : "dates",
    "Age"                  : "age_range",
    "Camp fee"             : "camp_fee",
    "Activities"           : "activities",
    "Family Stay"          : "family_stay",
    "Official Language"    : "official_language",
    "Camp Contact"         : "camp_contact",
    "Email"                : "email",
    "Phone"                : "phone",
    "Application Dead line": "application_deadline",
    "Website"              : "website",
}

def parse_camp_fields(raw_text,camp_name):
    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]

    data = {"camp_name": camp_name}

    i = 0
    while i < len(lines):
        line    = lines[i]
        matched = False
        for label, key in FIELD_MAP.items():
            if line.startswith(label):
                value_inline = line[len(label):].strip()
                if value_inline:
                    if key == "activities":
                        full_value = value_inline
                        j = i + 1
                        while j < len(lines) and not any(
                            lines[j].startswith(lbl) for lbl in FIELD_MAP
                        ):
                            full_value += " " + lines[j]
                            j += 1
                        data[key] = full_value.strip()
                        i = j
                    else:
                        data[key] = value_inline
                        i += 1
                elif i + 1 < len(lines):
                    data[key] = lines[i + 1]
                    i += 2
                else:
                    data[key] = ""
                    i += 1
                matched = True
                break
        if not matched:
            i += 1

    # Enrichissement dates
    if "dates" in data:
        parts = re.split(r"\s*[\.\-]\s*", data["dates"])
        if len(parts) == 2:
            data["date_start"] = parts[0].strip()
            data["date_end"]   = parts[1].strip()

    # Enrichissement âge
    if "age_range" in data:
        ages = re.split(r"[/\-]", data["age_range"])
        if len(ages) == 2:
            try:
                data["age_min"] = int(ages[0].strip())
                data["age_max"] = int(ages[1].strip())
            except ValueError:
                pass

    # Enrichissement fee
    if "camp_fee" in data:
        fee_num = re.sub(r"[^\d\.]", "", data["camp_fee"])
        data["camp_fee_eur"] = float(fee_num) if fee_num else None

    return data

In [ ]:
# ── 11. Scraping des liens PDF ────────────────────────────────
print("🌐 Scraping de la page principale...")
resp = requests.get(BASE_URL, timeout=15)
soup = BeautifulSoup(resp.text, "html.parser")

pdf_links = [
    (a.text.strip(), urljoin(BASE_DOMAIN, a["href"]))
    for a in soup.find_all("a", href=True)
    if ".pdf" in a["href"].lower()
]

print(f"🔗 {len(pdf_links)} liens PDF trouvés\n")

# ── 12. Traitement PDF à la volée ─────────────────────────────
results  = []
skipped  = []
errors   = []
log_rows = []

for idx, (link_text, pdf_url) in enumerate(pdf_links, 1):

    pdf_name = pdf_url.split("/")[-1]
    print(f"[{idx:>3}/{len(pdf_links)}] ⚙️  {pdf_name}", end="...")

    try:

        # ── Nom du camp + année depuis le lien HTML ───────────
        camp_name, year_from_link = parse_link_text(link_text, pdf_name)

        # ── Pays + région + state + année depuis l'URL ────────
        country, region, state, year_from_url = guess_country_region(pdf_url)

        # ── Année finale : URL prioritaire, lien en fallback ──
        year = year_from_link or year_from_url

        #if country != "Italy":
        # Filtre AVANT download pour tous les autres années
        if year != TARGET_YEAR:
             print(f"⏭️ {camp_name} / {country} / {year} ignoré ")
             skipped.append({"file": pdf_name, "camp_name": camp_name, "year": year})
             log_rows.append([camp_name, year or "?", "⏭️ ignoré", "", ""])
             continue

        # Téléchargement en mémoire
        r = requests.get(pdf_url, timeout=15)
        r.raise_for_status()
        pdf_bytes = r.content

        raw_text = extract_text_from_bytes(pdf_bytes)

        # ── Pour l'Italie : détection année depuis le texte ───
        #if country == "Italy":
        #    year = extract_year_from_pdf(raw_text)

        # Parsing champs
        camp_data = parse_camp_fields(raw_text, camp_name)

        record = {
            "source_url" : pdf_url,
            "pdf_file"   : pdf_name,
            "year"       : year,
            "region"     : region,
            "country"    : country,
            **({"state": state} if state else {}),
            **camp_data
        }
        results.append(record)

        # Sauvegarde PDF sur disque uniquement si 2026
        pdf_path = os.path.join(PDF_FOLDER, pdf_name)
        with open(pdf_path, "wb") as f:
            f.write(pdf_bytes)

        country_display = f"{country} ({state})" if state else country
        print(f"✅  {camp_name} / {year} / {country_display} / {region}")
        log_rows.append([camp_name, year, "✅ ok", country_display, region])

    except Exception as e:
        print(f"❌  {e}")
        errors.append({"file": pdf_name, "url": pdf_url, "error": str(e)})
        log_rows.append([pdf_name, "?", f"❌ {e}", "", ""])

# ── 13. Résumé console ────────────────────────────────────────
print(f"\n{'═'*60}")
print(f"  ✅ {len(results):>3} camps 2026 extraits")
print(f"  ⏭️  {len(skipped):>3} PDFs ignorés (autre année)")
print(f"  ❌ {len(errors):>3} erreurs")
print(f"{'═'*60}")

# ── 14. Tableau de log ────────────────────────────────────────
df_log = pd.DataFrame(log_rows, columns=["Fichier","Année","Statut","Pays","Région"])
display(df_log.style
    .set_properties(**{'font-size':'11px','text-align':'left'})
    .set_table_styles([
        {'selector':'th','props':[('background-color','#1a3a5c'),('color','white')]},
        {'selector':'tr:nth-child(even)','props':[('background-color','#e8f0fe')]}
    ])
)

# ── 15. Tableau récap camps 2026 ──────────────────────────────
if results:
    KEY_DISPLAY = [
        "pdf_file","camp_name","country","state","region",
        "camp_location","date_start","date_end",
        "age_range","camp_fee_eur","email","application_deadline"
    ]
    rows = [{k: d.get(k,"—") for k in KEY_DISPLAY} for d in results]
    print("\n🏕️  CAMPS 2026 EXTRAITS :")
    display(pd.DataFrame(rows).style
        .set_properties(**{'font-size':'11px','text-align':'left',
                           'white-space':'pre-wrap'})
        .set_table_styles([
            {'selector':'th','props':[('background-color','#2e6da4'),('color','white')]},
            {'selector':'tr:nth-child(even)','props':[('background-color','#f0f4ff')]}
        ])
    )

# ── 16. Stats par région ──────────────────────────────────────
if results:
    df_regions = (
        pd.DataFrame(results)[["region","country"]]
        .groupby("region")
        .agg(
            nb_camps = ("country","count"),
            pays     = ("country", lambda x: ", ".join(sorted(set(x))))
        )
        .reset_index()
        .sort_values("nb_camps", ascending=False)
    )
    print("\n🌍 RÉPARTITION PAR RÉGION :")
    display(df_regions.style
        .set_properties(**{'font-size':'11px','text-align':'left'})
        .set_table_styles([
            {'selector':'th','props':[('background-color','#1a5c3a'),('color','white')]},
            {'selector':'tr:nth-child(even)','props':[('background-color','#e8f5ee')]}
        ])
    )

# ── 17. Sauvegarde JSON ───────────────────────────────────────
output = {
    "year"        : TARGET_YEAR,
    "source"      : BASE_URL,
    "total_camps" : len(results),
    "skipped"     : len(skipped),
    "errors"      : errors,
    "camps"       : results
}

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

print(f"\n💾 JSON sauvegardé : {OUTPUT_JSON}")
print(f"📁 PDFs 2026 dans  : {PDF_FOLDER}/")

# ── 18. Téléchargement JSON ───────────────────────────────────
from google.colab import files
files.download(OUTPUT_JSON)
print("✅ Done !")

🌐 Scraping de la page principale...
🔗 131 liens PDF trouvés

[  1/131] ⚙️  Sound-of-music.pdf...✅  Lions Youthcamp "Sound of Music" / 2026 / Austria / Europe
[  2/131] ⚙️  Sound.pdf...✅  Brochure / 2026 / Austria / Europe
[  3/131] ⚙️  Brochure.pdf...✅  Lions Youthcamp "Käsköpfle together" / 2026 / Austria / Europe
[  4/131] ⚙️  Brochure.pdf...✅  Lions Youthcamp "Vienna and around" / 2026 / Austria / Europe
[  5/131] ⚙️  AUSTRIA-Campinfo-2023.pdf...⏭️ AUSTRIA-Campinfo-2023 / Austria / 2023 ignoré 
[  6/131] ⚙️  Brussels.pdf...✅  Brussels Lions Youth Camp / 2026 / Belgium / Europe
[  7/131] ⚙️  Belgiium_Facts-in-brief-2025.pdf...⏭️ Belgiium_Facts-in-brief-2025 / Belgium / 2025 ignoré 
[  8/131] ⚙️  Cipro.pdf...⏭️ Lions International Youth Camp "Nicos Michael" / Cipro / 2023 ignoré 
[  9/131] ⚙️  Croazia-Letak-YEC-A5.pdf...⏭️ International Lions Youth Camp Discover Croatia "Istria" / Croatia / 2025 ignoré 
[ 10/131] ⚙️  Danimarca.pdf...✅  Lions Camp Hirtshals / 2026 / Denmark / Europe
[ 

,Fichier,Année,Statut,Pays,Région
0,"Lions Youthcamp ""Sound of Music""",2026,✅ ok,Austria,Europe
1,Brochure,2026,✅ ok,Austria,Europe
2,"Lions Youthcamp ""Käsköpfle together""",2026,✅ ok,Austria,Europe
3,"Lions Youthcamp ""Vienna and around""",2026,✅ ok,Austria,Europe
4,AUSTRIA-Campinfo-2023,2023,⏭️ ignoré,,
5,Brussels Lions Youth Camp,2026,✅ ok,Belgium,Europe
6,Belgiium_Facts-in-brief-2025,2025,⏭️ ignoré,,
7,"Lions International Youth Camp ""Nicos Michael""",2023,⏭️ ignoré,,
8,"International Lions Youth Camp Discover Croatia ""Istria""",2025,⏭️ ignoré,,
9,Lions Camp Hirtshals,2026,✅ ok,Denmark,Europe



🏕️  CAMPS 2026 EXTRAITS :


,pdf_file,camp_name,country,state,region,camp_location,date_start,date_end,age_range,camp_fee_eur,email,application_deadline
0,Sound-of-music.pdf,"Lions Youthcamp ""Sound of Music""",Austria,—,Europe,"4040 Linz, Elmbergweg 65, Austria",18 Jul,1 Aug 2026,16/22,200.000000,yce.austria@gmail.com,—
1,Sound.pdf,Brochure,Austria,—,Europe,—,—,—,—,—,—,—
2,Brochure.pdf,"Lions Youthcamp ""Käsköpfle together""",Austria,—,Europe,—,—,—,—,—,—,—
3,Brochure.pdf,"Lions Youthcamp ""Vienna and around""",Austria,—,Europe,—,—,—,—,—,—,—
4,Brussels.pdf,Brussels Lions Youth Camp,Belgium,—,Europe,Limburg,18 Jul,1 Aug 2026,17/21,180.000000,stijn.renier@renier.be,31 Mar 2026
5,Danimarca.pdf,Lions Camp Hirtshals,Denmark,—,Europe,—,—,—,—,—,—,—
6,Active-Living.pdf,Active Living - Green Thinking,Denmark,—,Europe,"Videbæk, Denmark",4,18 Jul 2026,17/21,100.000000,ycein.md@lions.dk,—
7,Catch-the-future.pdf,"Catch the Future between Hearth, Heaven and Sea",Denmark,—,Europe,"Hirtshals, Denmark",4,18 Jul 2026,17/21,100.000000,ycein.md@lions.dk,—
8,Estonia.pdf,"29th Estonian Lions Youth Camp ""EESTI VÄGI""",Estonia,—,Europe,—,—,—,—,—,—,—
9,Perigord.pdf,Lions Camp du Perigord Noir,France,—,Europe,—,—,—,—,—,—,—



🌍 RÉPARTITION PAR RÉGION :


,region,nb_camps,pays
2,Europe,40,"Austria, Belgium, Denmark, Estonia, France, Germany, Hungary, Iceland, Ireland, Lithuania, Netherlands, Norway, Poland, Romania, Serbia, Slovakia, Slovenia, Spain, Sweden, Switzerland, Turkey, United Kingdom"
1,Asia-Pacific,17,"Australia, China, India, Indonesia, Japan, Mongolia, Sri Lanka, Taiwan"
5,North America,14,"Canada, United States"
7,South America,2,Brazil
3,Middle America,1,Mexico
0,Africa,1,Namibia
4,Middle East,1,Israel
6,Other,1,Bangladesh



💾 JSON sauvegardé : lions_camps_2026.json
📁 PDFs 2026 dans  : /content/pdf_camps/


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Done !


In [ ]:
skipped

[{'file': 'AUSTRIA-Campinfo-2023.pdf',
  'camp_name': 'AUSTRIA-Campinfo-2023',
  'year': '2023'},
 {'file': 'Belgiium_Facts-in-brief-2025.pdf',
  'camp_name': 'Belgiium_Facts-in-brief-2025',
  'year': '2025'},
 {'file': 'Cipro.pdf',
  'camp_name': 'Lions International Youth Camp "Nicos Michael"',
  'year': '2023'},
 {'file': 'Croazia-Letak-YEC-A5.pdf',
  'camp_name': 'International Lions Youth Camp Discover Croatia "Istria"',
  'year': '2025'},
 {'file': 'Camp-A.pdf', 'camp_name': 'Camp-A', 'year': '2023'},
 {'file': 'Danimarca.pdf',
  'camp_name': 'Lions Camp Western Jutland',
  'year': None},
 {'file': 'Lions_D120_Estonia_YC_2025.pdf',
  'camp_name': 'Lions_D120_Estonia_YC_2025',
  'year': '2025'},
 {'file': 'Lakeland%20Vibes%20-%20Lions%20Youth%20Camp%202026.pdf',
  'camp_name': 'Lions Youth Camp - Lakeland Vibes',
  'year': None},
 {'file': 'Paris.pdf', 'camp_name': 'Paris', 'year': '2025'},
 {'file': 'Peace.pdf', 'camp_name': 'Peace', 'year': '2022'},
 {'file': 'beitostlen-2025.pd

In [ ]:
log_rows

[['Lions Youthcamp "Sound of Music"', '2026', '✅ ok', 'Austria', 'Europe'],
 ['Brochure', '2026', '✅ ok', 'Austria', 'Europe'],
 ['Lions Youthcamp "Käsköpfle together"', '2026', '✅ ok', 'Austria', 'Europe'],
 ['Lions Youthcamp "Vienna and around"', '2026', '✅ ok', 'Austria', 'Europe'],
 ['AUSTRIA-Campinfo-2023', '2023', '⏭️ ignoré', '', ''],
 ['Brussels Lions Youth Camp', '2026', '✅ ok', 'Belgium', 'Europe'],
 ['Belgiium_Facts-in-brief-2025', '2025', '⏭️ ignoré', '', ''],
 ['Lions International Youth Camp "Nicos Michael"',
  '2023',
  '⏭️ ignoré',
  '',
  ''],
 ['International Lions Youth Camp Discover Croatia "Istria"',
  '2025',
  '⏭️ ignoré',
  '',
  ''],
 ['Lions Camp Hirtshals', '2026', '✅ ok', 'Denmark', 'Europe'],
 ['Camp-A', '2023', '⏭️ ignoré', '', ''],
 ['Lions Camp Western Jutland', '?', '⏭️ ignoré', '', ''],
 ['Active Living - Green Thinking', '2026', '✅ ok', 'Denmark', 'Europe'],
 ['Catch the Future between Hearth, Heaven and Sea',
  '2026',
  '✅ ok',
  'Denmark',
  'Europ

In [ ]:
results

[{'source_url': 'https://www.scambigiovanili-lions.org/images/doc-campi-esteri/Austria/2026/Sound-of-music.pdf',
  'pdf_file': 'Sound-of-music.pdf',
  'year': '2026',
  'region': 'Europe',
  'country': 'Austria',
  'camp_name': 'Lions Youthcamp "Sound of Music"',
  'camp_location': '4040 Linz, Elmbergweg 65, Austria',
  'dates': '18 Jul . 1 Aug 2026',
  'age_range': '16/22',
  'camp_fee': '200 €',
  'activities': 'Singing (choir and solo), making music, jamming, experience of singing on stage by microphone, workshops about understanding of other cultures, religions and other Lions topics, teambuilding, inclusion, sports, outdoor and indoor games, meeting with Austrian folk musicians, "lumberjack games"..and finally our great "Final Concert for Peace"!',
  'family_stay': '11 - 18 Jul 2026',
  'official_language': 'English',
  'camp_contact': 'Friedrich Drobesh',
  'email': 'yce.austria@gmail.com',
  'phone': 'Application Dead line 31 Mar 2026',
  'website': 'www.lionscamp.at',
  'date_s

In [ ]:
pdf_url="https://www.scambigiovanili-lions.org/images/doc-campi-esteri/Danimarca/2026/Danimarca.pdf"

#https://www.scambigiovanili-lions.org/images/doc-campi-esteri/Giappone/2026/Giappone.pdf"
#pdf_url="https://www.scambigiovanili-lions.org/images/doc-campi-esteri/Turchia/2026/Turchia.pdf"
#pdf_url="https://www.scambigiovanili-lions.org/images/doc-campi-esteri/Austria/2026/Brochure.pdf"
#pdf_url="https://www.scambigiovanili-lions.org/images/doc-campi-esteri/Svizzera/2026/Svizzera.pdf"
guess_country_region(pdf_url)

('Denmark', 'Europe', None, '2026')